# 01 · DreamBooth fine-tuning with LoRA (PEFT guide)

Follows the Hugging Face PEFT guide:
https://huggingface.co/docs/peft/v0.6.0/task_guides/dreambooth_lora

It uses PEFT's own training script (`examples/lora_dreambooth/train_dreambooth.py`, the one with
`--use_lora` / `--lora_r`), trains a separate LoRA per concept, and then composes adapters with
PEFT's `add_weighted_adapter` — which is the weighted-sum (normalized linear arithmetic) baseline
your thesis compares MoLE / CLoRA / LoRAtorio against.

Everything lives in Google Drive under this layout:

```
MyDrive/research/
├── external_repos/peft/                 # cloned PEFT repo (don't push to GitHub)
└── lora-dreambooth-project/             # your repo (push this)
    ├── notebooks/01_dreambooth_lora.ipynb
    ├── scripts/train_dreambooth_lora.py # copied from the PEFT example
    ├── configs/cat_config.yaml
    ├── datasets/   (don't push)
    └── outputs/    (don't push)
```

**Before running:** `Runtime → Change runtime type → T4 GPU`.

> Heads-up: this PEFT example predates the current diffusers/peft APIs. The notebook installs PEFT
> from the cloned repo so the script and library match. If a cell still errors on a library change,
> the diffusers-script notebook from earlier is the more actively maintained fallback.


## 0 · Check the GPU

In [1]:
!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available(), "| torch:", torch.__version__)

/bin/bash: line 1: nvidia-smi: command not found
CUDA: False | torch: 2.11.0+cpu


## 1 · Mount Drive and create the project structure

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os

ROOT      = "/content/drive/MyDrive/research"
PEFT_DIR  = os.path.join(ROOT, "external_repos", "peft")
PROJ      = os.path.join(ROOT, "lora-dreambooth-project")

DIRS = [
    os.path.join(ROOT, "external_repos"),
    os.path.join(PROJ, "notebooks"),
    os.path.join(PROJ, "scripts"),
    os.path.join(PROJ, "configs"),
    os.path.join(PROJ, "datasets", "dreambooth"),
    os.path.join(PROJ, "datasets", "class_images"),
    os.path.join(PROJ, "outputs"),
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print("Project root:", PROJ)
for d in DIRS:
    print("  ", d.replace(ROOT + "/", ""))

Mounted at /content/drive
Project root: /content/drive/MyDrive/research/lora-dreambooth-project
   external_repos
   lora-dreambooth-project/notebooks
   lora-dreambooth-project/scripts
   lora-dreambooth-project/configs
   lora-dreambooth-project/datasets/dreambooth
   lora-dreambooth-project/datasets/class_images
   lora-dreambooth-project/outputs


## 2 · Clone the PEFT repo into `external_repos/peft`
Skipped automatically if it already exists.

In [3]:
if not os.path.exists(os.path.join(PEFT_DIR, "setup.py")):
    !git clone --depth 1 https://github.com/huggingface/peft "{PEFT_DIR}"
else:
    print("PEFT already cloned at", PEFT_DIR)

EXAMPLE = os.path.join(PEFT_DIR, "examples", "lora_dreambooth", "train_dreambooth.py")
print("Training script present:", os.path.exists(EXAMPLE))

PEFT already cloned at /content/drive/MyDrive/research/external_repos/peft
Training script present: True


## 3 · Install dependencies and copy the training script

PEFT is installed **from the clone** so the installed library matches the script version. The script
is copied into your project as `scripts/train_dreambooth_lora.py` (so your repo is self-contained).

In [4]:
# install PEFT from the cloned repo + the libs the example needs
!pip install -q "{PEFT_DIR}"
!pip install -q "transformers>=4.41" "accelerate>=0.31" "diffusers>=0.27" safetensors ftfy

# Colab ships an old torchao (0.10) that current PEFT rejects with an ImportError inside
# get_peft_model -- even though LoRA training never uses torchao. Removing it makes PEFT treat
# it as simply unavailable (clean skip) instead of raising.
!pip uninstall -y torchao

import shutil
SCRIPT = os.path.join(PROJ, "scripts", "train_dreambooth_lora.py")
shutil.copy(EXAMPLE, SCRIPT)
print("Copied script ->", SCRIPT)

import peft, transformers, diffusers, accelerate
print("peft", peft.__version__, "| transformers", transformers.__version__,
      "| diffusers", diffusers.__version__, "| accelerate", accelerate.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Copied script -> /content/drive/MyDrive/research/lora-dreambooth-project/scripts/train_dreambooth_lora.py
peft 0.19.2.dev0 | transformers 5.10.2 | diffusers 0.38.0 | accelerate 1.13.0


## 4 · Download the subject images

Pulls subjects from the **DreamBooth dataset** (`google/dreambooth`) straight into your `datasets/`
folder. `cat`, `dog`, and `backpack` all live there.

Note: `wolf_plushie` is **not** in this dataset — the plushie/toy subjects come from
**CustomConcept101** (Custom Diffusion, Kumari et al.), at `github.com/adobe-research/custom-diffusion`.

In [5]:
from huggingface_hub import snapshot_download

SUBJECTS = ["cat", "dog", "backpack"]
DB_LOCAL = os.path.join(PROJ, "datasets", "dreambooth")

snapshot_download(
    "google/dreambooth", repo_type="dataset",
    allow_patterns=[f"dataset/{s}/*" for s in SUBJECTS] + ["dataset/prompts_and_classes.txt"],
    local_dir=DB_LOCAL,
)
for s in SUBJECTS:
    p = os.path.join(DB_LOCAL, "dataset", s)
    n = len([f for f in os.listdir(p) if f.lower().endswith((".jpg", ".jpeg", ".png"))]) if os.path.exists(p) else 0
    print(f"{s}: {n} images -> {p}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

cat: 5 images -> /content/drive/MyDrive/research/lora-dreambooth-project/datasets/dreambooth/dataset/cat
dog: 5 images -> /content/drive/MyDrive/research/lora-dreambooth-project/datasets/dreambooth/dataset/dog
backpack: 6 images -> /content/drive/MyDrive/research/lora-dreambooth-project/datasets/dreambooth/dataset/backpack


## 5 · Parse `prompts_and_classes.txt`, then write a config per concept

First we read the dataset's metadata file. It supplies two things: the correct **class noun** per
subject (the folder name isn't always the class — `wolf_plushie` → `stuffed animal`,
`colorful_sneaker` → `sneaker`), and the official **evaluation prompts** (separate object and
live-subject lists). The class map makes the prompts correct for *any* subject you pick; the eval
prompts are used at inference in Step 7.

Then we capture each concept's settings in a small YAML (`cat_config.yaml`, `dog_config.yaml`). Both
are written because the composition demo in Step 8 combines two trained LoRAs. Hyperparameters follow
the guide, with steps and class-image count trimmed for a reasonable Colab runtime (the guide uses
`max_train_steps=800`, `num_class_images=200`).

In [6]:
import re

# locate the metadata file (downloaded with the dataset, or uploaded to the session)
PROMPTS_FILE = os.path.join(PROJ, "datasets", "dreambooth", "dataset", "prompts_and_classes.txt")
if not os.path.exists(PROMPTS_FILE):
    for alt in ["/content/prompts_and_classes.txt", "prompts_and_classes.txt"]:
        if os.path.exists(alt):
            PROMPTS_FILE = alt
            break

LIVE_CLASSES = {"cat", "dog"}   # the dataset's 9 live subjects are cats and dogs

def parse_prompts_and_classes(path):
    text = open(path).read()
    # subject -> class noun
    subject_class = {}
    for line in text.split("Prompts")[0].splitlines():
        line = line.strip()
        if "," in line and not line.startswith("subject_name"):
            name, _, klass = line.partition(",")
            name, klass = name.strip(), klass.strip()
            if name and klass:
                subject_class[name] = klass
    # the two evaluation prompt template lists
    def block_after(header):
        i = text.find(header)
        if i == -1:
            return []
        seg = text[i:]
        seg = seg[seg.find("["): seg.find("]")]
        return re.findall(r"'([^']*)'\.format", seg)
    return subject_class, block_after("Object Prompts"), block_after("Live Subject Prompts")

SUBJECT_CLASS, OBJECT_PROMPTS, LIVE_PROMPTS = parse_prompts_and_classes(PROMPTS_FILE)
print("parsed", len(SUBJECT_CLASS), "subjects from", PROMPTS_FILE)
print("sample classes:", {k: SUBJECT_CLASS[k] for k in list(SUBJECT_CLASS)[:6]})
print("eval prompts -> object:", len(OBJECT_PROMPTS), "| live:", len(LIVE_PROMPTS))

def eval_prompts_for(subject, token="sks", n=None):
    klass = SUBJECT_CLASS.get(subject, subject)
    templates = LIVE_PROMPTS if klass in LIVE_CLASSES else OBJECT_PROMPTS
    out = [t.format(token, klass) for t in templates]
    return out[:n] if n else out

parsed 30 subjects from /content/drive/MyDrive/research/lora-dreambooth-project/datasets/dreambooth/dataset/prompts_and_classes.txt
sample classes: {'backpack': 'backpack', 'backpack_dog': 'backpack', 'bear_plushie': 'stuffed animal', 'berry_bowl': 'bowl', 'can': 'can', 'candle': 'candle'}
eval prompts -> object: 25 | live: 25


In [7]:
import yaml

MODEL_NAME = "CompVis/stable-diffusion-v1-4"   # guide default; still live on HF
# alt if you prefer SD 1.5: "stable-diffusion-v1-5/stable-diffusion-v1-5"

def make_config(concept, identifier="sks"):
    klass = SUBJECT_CLASS.get(concept, concept)   # correct class noun from prompts_and_classes.txt
    return {
        "concept": concept,
        "class": klass,
        "pretrained_model_name_or_path": MODEL_NAME,
        "instance_data_dir": os.path.join(PROJ, "datasets", "dreambooth", "dataset", concept),
        "class_data_dir":    os.path.join(PROJ, "datasets", "class_images", concept),
        "output_dir":        os.path.join(PROJ, "outputs", f"{concept}_lora"),
        "instance_prompt":   f"a photo of {identifier} {klass}",
        "class_prompt":      f"a photo of {klass}",
        "identifier":        identifier,
        "resolution": 512,
        "train_batch_size": 1,
        "num_class_images": 50,
        "max_train_steps": 500,
        "learning_rate": 1e-4,
        "lora_r": 16,
        "lora_alpha": 27,
        "lora_text_encoder_r": 16,
        "lora_text_encoder_alpha": 17,
    }

for concept in ["cat", "dog"]:
    cfg = make_config(concept)
    path = os.path.join(PROJ, "configs", f"{concept}_config.yaml")
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print("wrote", path)

# load the cat config to show it
with open(os.path.join(PROJ, "configs", "cat_config.yaml")) as f:
    print(f.read())

wrote /content/drive/MyDrive/research/lora-dreambooth-project/configs/cat_config.yaml
wrote /content/drive/MyDrive/research/lora-dreambooth-project/configs/dog_config.yaml
concept: cat
class: cat
pretrained_model_name_or_path: CompVis/stable-diffusion-v1-4
instance_data_dir: /content/drive/MyDrive/research/lora-dreambooth-project/datasets/dreambooth/dataset/cat
class_data_dir: /content/drive/MyDrive/research/lora-dreambooth-project/datasets/class_images/cat
output_dir: /content/drive/MyDrive/research/lora-dreambooth-project/outputs/cat_lora
instance_prompt: a photo of sks cat
class_prompt: a photo of cat
identifier: sks
resolution: 512
train_batch_size: 1
num_class_images: 50
max_train_steps: 500
learning_rate: 0.0001
lora_r: 16
lora_alpha: 27
lora_text_encoder_r: 16
lora_text_encoder_alpha: 17



## 6 · Train a LoRA for each concept

`accelerate launch` runs on the single Colab GPU with no extra setup. With `--with_prior_preservation`
the script first generates the class images into `class_data_dir`, then trains. Each concept saves
adapters into `outputs/<concept>_lora/{unet,text_encoder}`.

Expect roughly 15–25 min per concept on a T4 (class-image generation included).

In [8]:
import shlex, glob, yaml

def train_from_config(cfg_path):
    with open(cfg_path) as f:
        cfg = yaml.safe_load(f)
    args = [
        "accelerate", "launch", SCRIPT,
        f"--pretrained_model_name_or_path={cfg['pretrained_model_name_or_path']}",
        f"--instance_data_dir={cfg['instance_data_dir']}",
        f"--class_data_dir={cfg['class_data_dir']}",
        f"--output_dir={cfg['output_dir']}",
        "--train_text_encoder",
        "--with_prior_preservation", "--prior_loss_weight=1.0",
        f"--instance_prompt={cfg['instance_prompt']}",
        f"--class_prompt={cfg['class_prompt']}",
        f"--resolution={cfg['resolution']}",
        f"--train_batch_size={cfg['train_batch_size']}",
        "--lr_scheduler=constant", "--lr_warmup_steps=0",
        f"--num_class_images={cfg['num_class_images']}",
        "--use_lora",
        f"--lora_r={cfg['lora_r']}",
        f"--lora_alpha={cfg['lora_alpha']}",
        f"--lora_text_encoder_r={cfg['lora_text_encoder_r']}",
        f"--lora_text_encoder_alpha={cfg['lora_text_encoder_alpha']}",
        f"--learning_rate={cfg['learning_rate']}",
        "--gradient_accumulation_steps=1",
        "--gradient_checkpointing",
        "--mixed_precision=fp16",
        f"--max_train_steps={cfg['max_train_steps']}",
        "--seed=42",
    ]
    print("\n=== training:", cfg["instance_prompt"], "===\n")
    get_ipython().system(shlex.join(args))

for concept in ["cat", "dog"]:
    train_from_config(os.path.join(PROJ, "configs", f"{concept}_config.yaml"))


=== training: a photo of sks cat ===

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `0`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
06/17/2026 18:33:24 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cpu

Mixed precision type: fp16

06/17/2026 18:33:24 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/CompVis/st

In [9]:
# confirm adapters were saved
for concept in ["cat", "dog"]:
    base = os.path.join(PROJ, "outputs", f"{concept}_lora")
    ok = os.path.exists(os.path.join(base, "unet"))
    print(("OK  " if ok else "MISSING ") + base + "  (unet, text_encoder subdirs)")

MISSING /content/drive/MyDrive/research/lora-dreambooth-project/outputs/cat_lora  (unet, text_encoder subdirs)
MISSING /content/drive/MyDrive/research/lora-dreambooth-project/outputs/dog_lora  (unet, text_encoder subdirs)


## 7 · Single-adapter inference

This is the guide's `get_lora_sd_pipeline` helper: it rebuilds a Stable Diffusion pipeline and loads
the LoRA adapters from the saved `unet/` and `text_encoder/` folders. The prompts come straight from
the dataset's evaluation list for this subject (via `eval_prompts_for`), so it's the standard
recontextualization test rather than an ad-hoc prompt.

In [10]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline
from peft import PeftModel, LoraConfig

def get_lora_sd_pipeline(ckpt_dir, base_model_name_or_path=None, dtype=torch.float16,
                         device="cuda", adapter_name="default"):
    unet_sub = os.path.join(ckpt_dir, "unet")
    te_sub   = os.path.join(ckpt_dir, "text_encoder")
    if os.path.exists(te_sub) and base_model_name_or_path is None:
        base_model_name_or_path = LoraConfig.from_pretrained(te_sub).base_model_name_or_path
    if base_model_name_or_path is None:
        raise ValueError("Please specify the base model name or path")

    pipe = StableDiffusionPipeline.from_pretrained(base_model_name_or_path, torch_dtype=dtype,
                                                   safety_checker=None).to(device)
    pipe.unet = PeftModel.from_pretrained(pipe.unet, unet_sub, adapter_name=adapter_name)
    if os.path.exists(te_sub):
        pipe.text_encoder = PeftModel.from_pretrained(pipe.text_encoder, te_sub, adapter_name=adapter_name)
    if dtype in (torch.float16, torch.bfloat16):
        pipe.unet.half(); pipe.text_encoder.half()
    return pipe.to(device)

def show(images, titles):
    import matplotlib.pyplot as plt
    n = len(images); fig, ax = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1: ax = [ax]
    for a, im, t in zip(ax, images, titles):
        a.imshow(im); a.axis("off"); a.set_title(t)
    plt.tight_layout(); plt.show()

pipe = get_lora_sd_pipeline(os.path.join(PROJ, "outputs", "cat_lora"),
                            base_model_name_or_path=MODEL_NAME, adapter_name="cat")

# use the dataset's official evaluation prompts for this subject (cat -> live-subject list)
prompts = eval_prompts_for("cat", token="sks", n=3)
imgs = [pipe(p, num_inference_steps=40, guidance_scale=7.5,
             generator=torch.manual_seed(i)).images[0] for i, p in enumerate(prompts)]
show(imgs, prompts)
for p in prompts:
    print(p)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


AssertionError: Torch not compiled with CUDA enabled

## 8 · Multi-adapter composition  ← the thesis-relevant part

The guide's composition helpers: `load_adapter` adds a second LoRA, and `add_weighted_adapter`
merges adapters with chosen weights into a new named adapter. A weighted sum of adapters is exactly
the normalized-linear-arithmetic baseline (Eq. 2 in the MoLE paper).

Both concepts share the `sks` identifier here (as the guide does), so watch for concept bleed — it's
one of the failure modes your comparative study is meant to characterize. Using distinct identifier
tokens per concept reduces it.

In [ ]:
def load_adapter(pipe, ckpt_dir, adapter_name):
    pipe.unet.load_adapter(os.path.join(ckpt_dir, "unet"), adapter_name=adapter_name)
    te_sub = os.path.join(ckpt_dir, "text_encoder")
    if os.path.exists(te_sub) and isinstance(pipe.text_encoder, PeftModel):
        pipe.text_encoder.load_adapter(te_sub, adapter_name=adapter_name)

def set_adapter(pipe, adapter_name):
    pipe.unet.set_adapter(adapter_name)
    if isinstance(pipe.text_encoder, PeftModel):
        pipe.text_encoder.set_adapter(adapter_name)

def create_weighted_lora_adapter(pipe, adapters, weights, adapter_name="default"):
    pipe.unet.add_weighted_adapter(adapters, weights, adapter_name)
    if isinstance(pipe.text_encoder, PeftModel):
        pipe.text_encoder.add_weighted_adapter(adapters, weights, adapter_name)
    return pipe

# add the dog adapter alongside the already-loaded cat adapter
load_adapter(pipe, os.path.join(PROJ, "outputs", "dog_lora"), adapter_name="dog")

# merge cat + dog into one composed adapter, then generate
create_weighted_lora_adapter(pipe, ["cat", "dog"], [1.0, 1.0], adapter_name="cat_dog")
set_adapter(pipe, "cat_dog")

prompt = "a photo of sks cat and sks dog sitting together"
img = pipe(prompt, num_inference_steps=40, guidance_scale=7.5,
           generator=torch.manual_seed(1)).images[0]
show([img], ["cat + dog (1.0, 1.0)"])
print("Prompt:", prompt)

## 9 · Write `README.md`, `requirements.txt`, and `.gitignore` into the project

In [ ]:
readme = "\n".join([
    "# lora-dreambooth-project",
    "",
    "DreamBooth + LoRA fine-tuning via the Hugging Face PEFT example, run on Colab.",
    "Trains one LoRA per concept and composes adapters with PEFT `add_weighted_adapter`",
    "(the weighted-sum / normalized-linear-arithmetic baseline for the Multi-LoRA Composition thesis).",
    "",
    "## Run",
    "Open `notebooks/01_dreambooth_lora.ipynb` in Colab (T4 GPU) and run top to bottom.",
    "",
    "## Layout",
    "- `notebooks/` pipeline notebook",
    "- `scripts/train_dreambooth_lora.py` copied from PEFT `examples/lora_dreambooth`",
    "- `configs/` one YAML per concept",
    "- `datasets/`, `outputs/` generated at runtime (git-ignored)",
    "",
    "## Notes",
    "- Base model: `CompVis/stable-diffusion-v1-4`.",
    "- Subject images from the DreamBooth dataset (`google/dreambooth`).",
    "- PEFT cloned to `../external_repos/peft` (not part of this repo).",
])

requirements = "\n".join([
    "transformers>=4.41", "accelerate>=0.31", "diffusers>=0.27",
    "peft", "safetensors", "ftfy", "pyyaml", "huggingface_hub", "",
])

gitignore = "\n".join([
    "datasets/", "outputs/", "__pycache__/", "*.pyc",
    ".ipynb_checkpoints/", "*.safetensors", "",
])

for name, content in [("README.md", readme), ("requirements.txt", requirements), (".gitignore", gitignore)]:
    with open(os.path.join(PROJ, name), "w") as f:
        f.write(content)
    print("wrote", os.path.join(PROJ, name))

## Next steps toward the thesis

- This whole flow is the **baseline**. MoLE / CLoRA / LoRAtorio replace the fixed weighted sum in
  Step 8 with a learned or attention-driven composition — having the baseline running gives you the
  thing to compare against.
- Swap the DreamBooth subjects for **HAM10000** lesion crops. For the ADLC idea, train one LoRA per
  **ABCDE attribute** rather than per diagnosis class.
- Give each concept a **distinct identifier token** (not a shared `sks`) once you move past the
  reproduction step — it materially reduces concept bleed during composition.
- Plan evaluation early: CLIP image/text alignment (as in the paper), plus FID and a downstream
  classifier-accuracy check on real vs. augmented data.

To push to GitHub: commit only `lora-dreambooth-project/` (the `.gitignore` already excludes
`datasets/` and `outputs/`); leave `external_repos/peft/` out entirely.
